# Structured Data Exploitation Zone

This notebook transforms the cleaned Trusted Zone tables in ClickHouse into exploitation-ready dimensional models and analytical marts.

The design follows two rules:

- `natural_disaster_tweets`: text-oriented feature extraction.
- `global_warming_dataset`, `temperature_change`, and `co2_emission_by_vehicles`: dimensional modelling plus denormalized analytical marts.

## 1. Environment Setup

In [1]:
# Import the ClickHouse client used to create exploitation-zone tables.
import clickhouse_connect
import pandas as pd

# Source database: cleaned and standardized Trusted Zone tables.
TRUSTED_DB = "bi_analytics"

# Target database: curated tables for analytics, dashboards, and ML features.
EXPLOITATION_DB = "exploitation_analytics"

# Create one reusable ClickHouse connection for the whole notebook.
client = clickhouse_connect.get_client(
    host="clickhouse",
    port=8123,
    username="analytics",
    password="analytics_secret",
)

# Create the exploitation database if this notebook is executed for the first time.
client.command(f"CREATE DATABASE IF NOT EXISTS {EXPLOITATION_DB}")
print(f"Connected to ClickHouse. Target database: {EXPLOITATION_DB}")

Connected to ClickHouse. Target database: exploitation_analytics


## 2. Trusted Zone Validation

In [2]:
# These are the cleaned structured tables produced by the Trusted Zone notebook.
trusted_tables = [
    "natural_disaster_tweets",
    "global_warming_dataset",
    "temperature_change",
    "co2_emission_by_vehicles",
]

# Validate that every expected trusted table exists and contains rows before modelling.
for table_name in trusted_tables:
    row_count = client.query(f"SELECT count() FROM {TRUSTED_DB}.{table_name}").first_row[0]
    print(f"{table_name:<32} {row_count:>10,} rows")

natural_disaster_tweets             127,527 rows
global_warming_dataset              100,000 rows
temperature_change                    9,656 rows
co2_emission_by_vehicles              5,988 rows


## 3. Helper Functions

In [3]:
# This helper makes every table creation idempotent: rerunning the notebook replaces old outputs.
def recreate_table(table_name: str, select_sql: str, order_by: str) -> None:
    """Create an exploitation table from a SELECT statement in an idempotent way."""
    full_name = f"{EXPLOITATION_DB}.{table_name}"
    client.command(f"DROP TABLE IF EXISTS {full_name}")
    client.command(
        f"""
        CREATE TABLE {full_name}
        ENGINE = MergeTree()
        ORDER BY {order_by}
        SETTINGS allow_nullable_key = 1
        AS
        {select_sql}
        """
    )
    row_count = client.query(f"SELECT count() FROM {full_name}").first_row[0]
    print(f"Created {full_name:<55} {row_count:>10,} rows")

## 4. Dimension Tables

Dimension tables describe the main analytical entities used by the fact tables and marts. They make the model easier to query because repeated descriptive values are centralized into reusable lookup tables.

| Table | Grain | Purpose |
|---|---|---|
| `dim_year` | One row per year | Provides a shared time dimension and adds `decade` for long-term trend analysis. |
| `dim_country` | One row per country from the climate dataset | Connects country-level climate facts to readable country names. |
| `dim_area` | One row per temperature-change area | Describes geographic areas in the temperature dataset, including the M49 area code. |
| `dim_month` | One row per month label/code | Supports monthly temperature trend analysis. |
| `dim_disaster_type` | One row per disaster category | Standardizes tweet disaster categories such as flood, earthquake, or wildfire. |
| `dim_vehicle` | One row per vehicle configuration | Describes make, model, class, engine, transmission, and fuel attributes for vehicle-emission analysis. |

In [6]:
# Build reusable dimensions for years, countries, areas, months, disaster types, and vehicles.
# Surrogate keys are generated with cityHash64 where the source does not provide a stable numeric key.
recreate_table(
    "dim_year",
    f"""
    SELECT DISTINCT
        toInt32(Year) AS year_key,
        toInt32(Year) AS year,
        intDiv(toInt32(Year), 10) * 10 AS decade
    FROM {TRUSTED_DB}.global_warming_dataset
    UNION DISTINCT
    SELECT DISTINCT
        toInt32(Year) AS year_key,
        toInt32(Year) AS year,
        intDiv(toInt32(Year), 10) * 10 AS decade
    FROM {TRUSTED_DB}.temperature_change
    """,
    "year_key",
)

recreate_table(
    "dim_country",
    f"""
    SELECT DISTINCT
        cityHash64(Country) AS country_key,
        Country AS country_name
    FROM {TRUSTED_DB}.global_warming_dataset
    WHERE Country != ''
    """,
    "country_key",
)

recreate_table(
    "dim_area",
    f"""
    SELECT DISTINCT
        cityHash64(Area) AS area_key,
        Area_Code_M49 AS area_code_m49,
        Area AS area_name
    FROM {TRUSTED_DB}.temperature_change
    WHERE Area != ''
    """,
    "area_key",
)

recreate_table(
    "dim_month",
    f"""
    SELECT DISTINCT
        Months_Code AS month_key,
        Months AS month_name
    FROM {TRUSTED_DB}.temperature_change
    WHERE Months != ''
    """,
    "month_key",
)

recreate_table(
    "dim_disaster_type",
    f"""
    SELECT DISTINCT
        cityHash64(disaster_type) AS disaster_type_key,
        disaster_type
    FROM {TRUSTED_DB}.natural_disaster_tweets
    WHERE disaster_type != ''
    """,
    "disaster_type_key",
)

recreate_table(
    "dim_vehicle",
    f"""
    SELECT DISTINCT
        cityHash64(Make, Model, Vehicle_Class, Transmission, Fuel_Type, Engine_SizeL, Cylinders) AS vehicle_key,
        Make AS make,
        Model AS model,
        Vehicle_Class AS vehicle_class,
        Engine_SizeL AS engine_size_l,
        Cylinders AS cylinders,
        Transmission AS transmission,
        Fuel_Type AS fuel_type
    FROM {TRUSTED_DB}.co2_emission_by_vehicles
    """,
    "vehicle_key",
)

Created exploitation_analytics.dim_disaster_type                         5 rows


## 5. Fact Tables

Fact tables store measurable events or observations at a clear grain. They keep the source data analytically structured while replacing repeated descriptive columns with dimension keys.

| Table | Grain | Purpose |
|---|---|---|
| `fact_climate_country_year` | One country per year | Stores climate, economy, population, emissions, policy, and environmental indicators for country-year analysis. |
| `fact_temperature_area_month` | One area per month per year | Stores monthly temperature-change values and quality flags for time-series trend analysis. |
| `fact_vehicle_emission` | One vehicle-emission record | Stores fuel-consumption and CO2-emission measurements linked to vehicle attributes. |

In [ ]:
# Build fact tables at the natural analytical grain of each trusted structured dataset.
# These tables are normalized enough for reuse, while marts later denormalize them for BI consumption.
recreate_table(
    "fact_climate_country_year",
    f"""
    SELECT
        cityHash64(Country) AS country_key,
        toInt32(Year) AS year_key,
        Temperature_Anomaly AS temperature_anomaly,
        Average_Temperature AS average_temperature,
        CO2_Emissions AS co2_emissions,
        Population AS population,
        Forest_Area AS forest_area,
        GDP AS gdp,
        Renewable_Energy_Usage AS renewable_energy_usage,
        Methane_Emissions AS methane_emissions,
        Sea_Level_Rise AS sea_level_rise,
        Arctic_Ice_Extent AS arctic_ice_extent,
        Urbanization AS urbanization,
        Deforestation_Rate AS deforestation_rate,
        Extreme_Weather_Events AS extreme_weather_events,
        Average_Rainfall AS average_rainfall,
        Solar_Energy_Potential AS solar_energy_potential,
        Waste_Management AS waste_management,
        Per_Capita_Emissions AS source_per_capita_emissions,
        Industrial_Activity AS industrial_activity,
        Air_Pollution_Index AS air_pollution_index,
        Biodiversity_Index AS biodiversity_index,
        Ocean_Acidification AS ocean_acidification,
        Fossil_Fuel_Usage AS fossil_fuel_usage,
        Energy_Consumption_Per_Capita AS energy_consumption_per_capita,
        Policy_Score AS policy_score
    FROM {TRUSTED_DB}.global_warming_dataset
    """,
    "(country_key, year_key)",
)

recreate_table(
    "fact_temperature_area_month",
    f"""
    SELECT
        cityHash64(Area) AS area_key,
        toInt32(Year) AS year_key,
        Months_Code AS month_key,
        Domain AS domain,
        Element AS element,
        Unit AS unit,
        Value AS temperature_change_value,
        Flag AS flag,
        Flag_Description AS flag_description
    FROM {TRUSTED_DB}.temperature_change
    """,
    "(area_key, year_key, month_key)",
)

recreate_table(
    "fact_vehicle_emission",
    f"""
    SELECT
        cityHash64(Make, Model, Vehicle_Class, Transmission, Fuel_Type, Engine_SizeL, Cylinders) AS vehicle_key,
        Fuel_Consumption_City_L_100_km AS fuel_consumption_city_l_100_km,
        Fuel_Consumption_Hwy_L_100_km AS fuel_consumption_hwy_l_100_km,
        Fuel_Consumption_Comb_L_100_km AS fuel_consumption_comb_l_100_km,
        Fuel_Consumption_Comb_mpg AS fuel_consumption_comb_mpg,
        CO2_Emissionsg_km AS co2_emissions_g_km
    FROM {TRUSTED_DB}.co2_emission_by_vehicles
    """,
    "vehicle_key",
)

## 6. Tweet Feature Extraction

Tweets are different from the other structured datasets because their main analytical value is inside free text. This section converts tweet text into structured features and separates multi-valued hashtags into a bridge table.

| Table | Grain | Purpose |
|---|---|---|
| `fact_tweet_features` | One row per tweet | Converts tweet text into numerical and categorical features such as word count, hashtag count, URL count, alert-keyword flag, and simple sentiment label. |
| `bridge_tweet_hashtag` | One row per tweet-hashtag pair | Resolves the one-to-many relationship between tweets and hashtags, making hashtag frequency and hashtag-disaster analysis possible. |

In [8]:
# Extract lightweight text features directly in ClickHouse.
# ifNull(tweet_text, '') prevents Nullable(String) values from producing invalid Nullable(Array) results.
recreate_table(
    "fact_tweet_features",
    f"""
    SELECT
        id AS tweet_id,
        cityHash64(disaster_type) AS disaster_type_key,
        disaster_type,
        ifNull(tweet_text, '') AS tweet_text,
        lengthUTF8(ifNull(tweet_text, '')) AS text_length,
        length(splitByChar(' ', trim(ifNull(tweet_text, '')))) AS word_count,
        countMatches(ifNull(tweet_text, ''), '#[A-Za-z0-9_]+') AS hashtag_count,
        countMatches(ifNull(tweet_text, ''), '@[A-Za-z0-9_]+') AS mention_count,
        countMatches(ifNull(tweet_text, ''), 'https?://|www\\.') AS url_count,
        countMatches(ifNull(tweet_text, ''), '!') AS exclamation_count,
        countMatches(ifNull(tweet_text, ''), '\\?') AS question_count,
        if(positionCaseInsensitive(ifNull(tweet_text, ''), 'youtube') > 0, 1, 0) AS contains_youtube,
        if(
            positionCaseInsensitive(ifNull(tweet_text, ''), 'emergency') > 0
            OR positionCaseInsensitive(ifNull(tweet_text, ''), 'warning') > 0
            OR positionCaseInsensitive(ifNull(tweet_text, ''), 'evacuation') > 0,
            1,
            0
        ) AS contains_alert_keyword,
        (
            if(positionCaseInsensitive(ifNull(tweet_text, ''), 'safe') > 0, 1, 0)
            + if(positionCaseInsensitive(ifNull(tweet_text, ''), 'rescue') > 0, 1, 0)
            + if(positionCaseInsensitive(ifNull(tweet_text, ''), 'help') > 0, 1, 0)
            - if(positionCaseInsensitive(ifNull(tweet_text, ''), 'damage') > 0, 1, 0)
            - if(positionCaseInsensitive(ifNull(tweet_text, ''), 'dead') > 0, 1, 0)
            - if(positionCaseInsensitive(ifNull(tweet_text, ''), 'destroyed') > 0, 1, 0)
        ) AS simple_sentiment_score,
        multiIf(simple_sentiment_score > 0, 'positive', simple_sentiment_score < 0, 'negative', 'neutral') AS simple_sentiment_label
    FROM {TRUSTED_DB}.natural_disaster_tweets
    """,
    "tweet_id",
)

Created exploitation_analytics.fact_tweet_features                 127,527 rows


In [9]:
# Split hashtags out of tweet text so each tweet-hashtag relationship becomes one analyzable row.
recreate_table(
    "bridge_tweet_hashtag",
    f"""
    SELECT
        id AS tweet_id,
        lowerUTF8(replaceRegexpAll(hashtag, '^#', '')) AS hashtag
    FROM {TRUSTED_DB}.natural_disaster_tweets
    ARRAY JOIN extractAll(ifNull(tweet_text, ''), '#[A-Za-z0-9_]+') AS hashtag
    WHERE hashtag != ''
    """,
    "(hashtag, tweet_id)",
)

Created exploitation_analytics.bridge_tweet_hashtag                192,677 rows


## 7. Denormalized Analytical Marts

Marts are query-friendly tables designed for dashboards, reports, and direct analysis. They intentionally denormalize dimensions and facts so users do not need to write many joins.

| Table | Grain | Purpose |
|---|---|---|
| `mart_climate_country_year` | One country per year | Dashboard-ready climate mart with derived metrics such as CO2 per capita, GDP per capita, and emission intensity. |
| `mart_temperature_trends` | One area per month per year | Time-series mart with annual average temperature change and rolling five-year monthly averages. |
| `mart_vehicle_emission_summary` | One make/model/class/fuel group | Summarizes average CO2 emissions and fuel consumption, with ranking inside each vehicle class. |
| `mart_disaster_tweet_features` | One row per tweet | ML/BI-ready tweet feature table without raw modelling joins. |
| `mart_disaster_type_profile` | One row per disaster type | Aggregated profile comparing tweet volume, text length, hashtag usage, alert-keyword share, and simple sentiment by disaster category. |

In [11]:
# Create dashboard-ready marts for climate, temperature, and vehicle-emission analysis.
# These marts join dimension labels back onto facts and add derived analytical metrics.
recreate_table(
    "mart_climate_country_year",
    f"""
    SELECT
        c.country_name AS country,
        y.year,
        y.decade,
        f.temperature_anomaly,
        f.average_temperature,
        f.co2_emissions,
        f.population,
        if(f.population = 0, NULL, f.co2_emissions / f.population) AS co2_per_capita,
        f.gdp,
        if(f.population = 0, NULL, f.gdp / f.population) AS gdp_per_capita,
        if(f.gdp = 0, NULL, f.co2_emissions / f.gdp) AS emission_intensity,
        f.renewable_energy_usage,
        f.forest_area,
        f.deforestation_rate,
        f.extreme_weather_events,
        f.policy_score,
        f.air_pollution_index,
        f.biodiversity_index,
        f.fossil_fuel_usage
    FROM {EXPLOITATION_DB}.fact_climate_country_year f
    INNER JOIN {EXPLOITATION_DB}.dim_country c USING country_key
    INNER JOIN {EXPLOITATION_DB}.dim_year y USING year_key
    """,
    "(country, year)",
)

# Create tweet-focused marts: one feature-level table for ML and one aggregated profile table for BI.
recreate_table(
    "mart_temperature_trends",
    f"""
    SELECT
        a.area_name AS area,
        y.year,
        m.month_name AS month,
        f.temperature_change_value,
        avg(f.temperature_change_value) OVER (PARTITION BY a.area_name, y.year) AS annual_avg_temperature_change,
        avg(f.temperature_change_value) OVER (
            PARTITION BY a.area_name, m.month_name
            ORDER BY y.year
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ) AS rolling_5y_month_avg
    FROM {EXPLOITATION_DB}.fact_temperature_area_month f
    INNER JOIN {EXPLOITATION_DB}.dim_area a USING area_key
    INNER JOIN {EXPLOITATION_DB}.dim_year y USING year_key
    INNER JOIN {EXPLOITATION_DB}.dim_month m USING month_key
    """,
    "(area, year, month)",
)

recreate_table(
    "mart_vehicle_emission_summary",
    f"""
    SELECT
        v.make,
        v.model,
        v.vehicle_class,
        v.fuel_type,
        count() AS vehicle_record_count,
        avg(f.co2_emissions_g_km) AS avg_co2_emissions_g_km,
        avg(f.fuel_consumption_comb_l_100_km) AS avg_fuel_consumption_comb_l_100_km,
        avg(f.fuel_consumption_comb_mpg) AS avg_fuel_consumption_comb_mpg,
        rank() OVER (PARTITION BY v.vehicle_class ORDER BY avg(f.co2_emissions_g_km) DESC) AS emission_rank_in_class
    FROM {EXPLOITATION_DB}.fact_vehicle_emission f
    INNER JOIN {EXPLOITATION_DB}.dim_vehicle v USING vehicle_key
    GROUP BY v.make, v.model, v.vehicle_class, v.fuel_type
    """,
    "(vehicle_class, emission_rank_in_class, make, model)",
)

In [12]:
# Create tweet-focused marts: one feature-level table for ML and one aggregated profile table for BI.
recreate_table(
    "mart_disaster_tweet_features",
    f"""
    SELECT
        tweet_id,
        disaster_type,
        text_length,
        word_count,
        hashtag_count,
        mention_count,
        url_count,
        exclamation_count,
        question_count,
        contains_youtube,
        contains_alert_keyword,
        simple_sentiment_score,
        simple_sentiment_label
    FROM {EXPLOITATION_DB}.fact_tweet_features
    """,
    "tweet_id",
)

recreate_table(
    "mart_disaster_type_profile",
    f"""
    SELECT
        disaster_type,
        count() AS tweet_count,
        avg(text_length) AS avg_text_length,
        avg(word_count) AS avg_word_count,
        avg(hashtag_count) AS avg_hashtag_count,
        avg(mention_count) AS avg_mention_count,
        avg(url_count) AS avg_url_count,
        avg(contains_alert_keyword) AS alert_keyword_share,
        avg(simple_sentiment_score) AS avg_simple_sentiment_score,
        countIf(simple_sentiment_label = 'positive') AS positive_tweets,
        countIf(simple_sentiment_label = 'negative') AS negative_tweets,
        countIf(simple_sentiment_label = 'neutral') AS neutral_tweets
    FROM {EXPLOITATION_DB}.fact_tweet_features
    GROUP BY disaster_type
    """,
    "disaster_type",
)

Created exploitation_analytics.mart_disaster_tweet_features        127,527 rows
Created exploitation_analytics.mart_disaster_type_profile                5 rows


## 8. Validation and Preview

In [13]:
# List every exploitation table and validate that each output contains the expected number of rows.
exploitation_tables = client.query(
    f"""
    SELECT name
    FROM system.tables
    WHERE database = '{EXPLOITATION_DB}'
    ORDER BY name
    """
).result_rows

for (table_name,) in exploitation_tables:
    row_count = client.query(f"SELECT count() FROM {EXPLOITATION_DB}.{table_name}").first_row[0]
    print(f"{table_name:<40} {row_count:>10,} rows")

bridge_tweet_hashtag                        192,677 rows
dim_country                                     195 rows
dim_disaster_type                                 5 rows
fact_climate_country_year                   100,000 rows
fact_tweet_features                         127,527 rows
mart_disaster_tweet_features                127,527 rows
mart_disaster_type_profile                        5 rows


In [ ]:
# Preview the main marts so the notebook output can be inspected immediately after execution.
preview_queries = {
    "mart_climate_country_year": "SELECT * FROM exploitation_analytics.mart_climate_country_year LIMIT 5",
    "mart_temperature_trends": "SELECT * FROM exploitation_analytics.mart_temperature_trends LIMIT 5",
    "mart_vehicle_emission_summary": "SELECT * FROM exploitation_analytics.mart_vehicle_emission_summary LIMIT 5",
    "mart_disaster_type_profile": "SELECT * FROM exploitation_analytics.mart_disaster_type_profile ORDER BY tweet_count DESC LIMIT 10",
}

for title, query in preview_queries.items():
    print(f"\n{title}")
    display(client.query_df(query))